In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Поиск 6 наиболее похожих изображений для каждого кадра.
Author: <ваше_имя>
"""

import os
from pathlib import Path
import torch
from torchvision import transforms
import pandas as pd
import faiss                     # pip install faiss-cpu
from tqdm import tqdm
from PIL import Image

# ─────────────────────────── 1. Параметры ────────────────────────────────
IMG_DIR   = Path("dataset")        # каталог с архивом рисунков
CSV_OUT   = "submission.csv"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
BATCH     = 32

# ─────────────────────────── 2. Модель CLIP ──────────────────────────────
import clip
model, preprocess = clip.load("ViT-B/32", device=DEVICE)   # 512-мерный эмбеддинг
model.eval()

# ─────────────────────────── 3. Препроцессинг ────────────────────────────
# clip.load даёт уже готовый transforms.Compose, но нам нужна batch-версия
tfm = preprocess                   # включает resize → 224, нормировку и т.д.

# ─────────────────────────── 4. Считываем изображения ────────────────────
paths = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
N = len(paths)
print(f"Found {N} images")

# ─────────────────────────── 5. Извлекаем эмбеддинги ─────────────────────
all_features = torch.zeros((N, 512), dtype=torch.float32, device=DEVICE)

with torch.inference_mode():
    for i in tqdm(range(0, N, BATCH)):
        batch_paths = paths[i:i+BATCH]
        imgs = [ tfm(Image.open(p).convert("RGB")) for p in batch_paths ]
        imgs = torch.stack(imgs).to(DEVICE)
        feats = model.encode_image(imgs)            # (B, 512)
        feats /= feats.norm(dim=-1, keepdim=True)   # L2-нормируем
        all_features[i:i+len(batch_paths)] = feats

all_features = all_features.cpu().numpy().astype('float32')   # FAISS ждёт float32

# ─────────────────────────── 6. FAISS-индекс (IP) ────────────────────────
index = faiss.IndexFlatIP(512)        # Inner Product
index.add(all_features)               # база = сами же изображения

# ─────────────────────────── 7. Поиск 7 соседей → топ-6 ──────────────────
k = 7                                 # включая самого себя
D, I = index.search(all_features, k)  # I.shape = (N, 7)

# ─────────────────────────── 8. Формируем submission.csv ─────────────────
records = []
for idx, (row_ids) in enumerate(I):
    # первый ID всегда idx (сам себе), пропускаем
    neigh = [paths[j].name for j in row_ids if j != idx][:6]
    records.append({"filename": paths[idx].name,
                    "ranking": " ".join(neigh)})

df = pd.DataFrame(records)
df.to_csv(CSV_OUT, index=False)
print(f"Saved → {CSV_OUT}")

Found 9605 images


100%|██████████| 301/301 [02:18<00:00,  2.17it/s]

: 